<a href="https://colab.research.google.com/github/guyen5/Capstone/blob/main/notebooks/4%20Replication%20Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4: Replication Run

This notebook replicates the two baselines of Wang and Hasuike (2026) on the project's data. **Plain PTO** forecasts each ETF's return with a ridge regression and passes the forecasts to the Markowitz optimizer. **Classic Markowitz** passes historical average returns to the same optimizer. Both run for every risk aversion λ in the paper's grid, on both universes, over the 81 test months, and the results are set against the paper's Table 3.

The run tests H1: plain PTO trades far more than classic Markowitz, and raising λ does not fix it. Every setting follows the Replication Spec, and the four pass criteria in the spec are checked in section 8. They were fixed before this run.

**Work package:** WBS 3.4 (Replication run)

**Inputs:** Daily Features, Daily Training Windows and Covariances from Notebook 3, Monthly Data from Notebook 2, Daily Prices from Notebook 1

**Outputs** (Google Drive, `Capstone/data`):

| File | Folder | Contents |
|---|---|---|
| PTO Forecasts.parquet | processed | Each ETF's forecast return at each decision date, with the chosen ridge penalty |
| Replication Months.parquet | processed | Monthly gross and net return, fee and turnover of plain PTO and classic Markowitz, every λ and universe |
| Replication Results.csv | reports | Return, volatility, Sharpe ratio, drawdown and turnover of each run, next to the paper's values |
| Replication Checks.csv | reports | Results of all checks, including the four pass criteria |

## 1. Setup

Connects Google Drive, where all data files are stored, and loads the project code from the GitHub repository. Run this cell first in every new Colab session.

In [ ]:
# Connect Google Drive. All data files are read from and saved to
# MyDrive/Capstone/data.
from google.colab import drive
drive.mount("/content/drive")

import os
import sys

REPO_URL = "https://github.com/guyen5/Capstone.git"
REPO_DIR = "/content/Capstone"

# Download the project code on the first run of a session; on later runs,
# replace it with the latest version on GitHub.
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet origin
    !git -C {REPO_DIR} reset --quiet --hard origin/main
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}

# Install the pinned library versions listed in requirements.txt.
%pip install --quiet -r {REPO_DIR}/requirements.txt

# Make the project modules importable with "from src import ...".
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

In [ ]:
import importlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config, cleaning, windows, covariance, variables, optimizer, backtest, forecast

# Reload the modules so code changes pulled from GitHub take effect
# without restarting the Colab runtime.
for module in (config, cleaning, windows, covariance, variables, optimizer, backtest, forecast):
    importlib.reload(module)

DIRS = config.data_dirs()
pd.set_option("display.width", 160)
checks = []

## 2. Load the inputs

The daily table and its training windows drive the forecasts; the holding-month returns, covariances and 126-day historical means drive the backtest. Only the paper's 9+3 window is used here.

In [ ]:
proc = DIRS["processed"]
clean = cleaning.load_clean(proc)
panel = pd.read_parquet(proc / config.FILES["monthly"])
daily_table = pd.read_parquet(proc / config.FILES["daily_table"])
all_windows = pd.read_parquet(proc / config.FILES["daily_windows"])
cov_table = pd.read_parquet(proc / config.FILES["covariances"])

calendar = all_windows[all_windows["window"] == "9+3"].reset_index(drop=True)
universes = ("ETF_A", "ETF_B")
returns = {u: backtest.holding_returns(panel, u) for u in universes}
cov_sample = {u: backtest.covariance_lookup(cov_table, u, "sample") for u in universes}
hist_mu = {u: backtest.historical_means(clean, returns[u].index, u) for u in universes}

assert (pd.DatetimeIndex(calendar["decision_date"]) == returns["ETF_A"].index).all()
print(f"{len(calendar)} decision dates, {calendar['month'].iloc[0]} to {calendar['month'].iloc[-1]}")
print(f"Classic Markowitz averages the last {config.MU_DAYS} trading days")

## 3. Forecasts

At each decision date a ridge regression is fitted on the daily rows of the 9+3 window and predicts each ETF's return over the coming month. As set out in the Replication Spec:

- **Inputs:** the seven features, standardized on the rows being fitted, plus a 0/1 column for each ticker. One model is shared by all ETFs in a universe.
- **Target:** the 21-day forward return.
- **Penalty:** one of 15 values from 0.1 to 10⁶, chosen by squared error on the 3-month validation slice. While tuning, only rows whose targets are known before validation starts are fitted. The chosen model is then refit on the whole 12 months and predicts at the decision date.

A large penalty pulls every forecast toward the same value, the average return of the window. A small one lets the features and tickers separate the ETFs.

In [ ]:
fc = pd.concat([forecast.build_forecasts(daily_table, calendar, u, backtest.tickers_of(u)) for u in universes],
               ignore_index=True)
per_month = fc.drop_duplicates(["universe", "date"])
print(f"{len(fc):,} forecasts, {per_month['fit_rows'].median():.0f} fit rows and "
      f"{per_month['validation_rows'].median():.0f} validation rows in a typical month")
print()
print("Chosen ridge penalty (share of months)")
print(per_month.groupby("universe")["alpha"].value_counts(normalize=True).unstack(0).fillna(0).round(2).to_string())

## 4. Checks on the forecasts

Confirms that every month has a forecast for every ETF, that no training target is later than its decision date, and that three random months refit with scikit-learn's `Ridge` give the same forecasts.

In [ ]:
for u in universes:
    report = forecast.check_forecasts(fc, daily_table, calendar, u, backtest.tickers_of(u))
    print(u)
    print(report.to_string(index=False))
    print()
    checks.append(report.assign(table=f"PTO Forecasts {u}"))
fc_columns = variables.check_table(fc, "PTO Forecasts")
checks.append(fc_columns.assign(table="PTO Forecasts"))
print("Column checks:", fc_columns["status"].value_counts().to_dict())
if (pd.concat(checks)["status"] == "FAIL").any():
    raise RuntimeError("A forecast check failed.")

## 5. How good are the forecasts

| Measure | Meaning |
|---|---|
| Out-of-sample R² | 1 minus squared forecast error relative to a forecast of zero. Monthly returns are mostly noise, so values near 0 are normal and negative values are common |
| Mean IC | Average monthly rank correlation between forecasts and realized returns across ETFs. 0 means no ranking skill |
| IC t-statistic | Mean IC divided by its standard error over the 81 months |
| Top pick beats average | Share of months in which the ETF with the highest forecast beat the universe average; 0.5 is chance |
| Spread ratio | Cross-sectional spread of forecasts divided by that of realized returns. Well below 1 means the forecasts are cautious |

The chart compares the average forecast across ETFs with the average realized return each month, the same view as the paper's Figure 1. The paper's PTO line stays close to zero, unlike its SPO line.

In [ ]:
quality = forecast.forecast_quality(fc, panel)
print(quality.round(3).to_string())

realized = panel[["date", "universe", "ticker", "next_return"]]
m = fc.merge(realized, on=["date", "universe", "ticker"])
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharey=True)
for ax, u in zip(axes, universes):
    g = m[m["universe"] == u].groupby("date")[["forecast", "next_return"]].mean()
    ax.plot(g.index, g["next_return"], color="black", lw=1, label="realized")
    ax.plot(g.index, g["forecast"], color="tab:red", lw=1.5, label="PTO forecast")
    ax.axhline(0, color="grey", lw=0.5)
    ax.set_title(f"Average across ETFs, {u}")
axes[0].set_ylabel("monthly return")
axes[0].legend()
plt.tight_layout()
plt.show()

## 6. The replication run

Both strategies use the same optimizer, the paper's sample covariance, κ = 0.005, a start from equal weights and the 0.5% fee. They differ only in the expected returns: the ridge forecasts for plain PTO, the 126-day historical means for classic Markowitz. Every month of every run is recomputed independently.

In [ ]:
mu_pto = {u: fc[fc["universe"] == u].pivot(index="date", columns="ticker", values="forecast")[backtest.tickers_of(u)]
          for u in universes}
runs, weights_by_run = [], {}
for u in universes:
    for name, mu in (("Plain PTO", mu_pto[u]), ("Classic Markowitz", hist_mu[u])):
        for lam in config.LAMBDA_GRID:
            mo, w = backtest.run(backtest.mean_variance(mu, cov_sample[u], lam), returns[u])
            report = backtest.check_accounting(mo, w, returns[u])
            checks.append(report.assign(table=f"Backtest {u} {name} {lam}"))
            runs.append(mo.assign(strategy=name, universe=u, lam=lam))
            weights_by_run[(name, u, lam)] = w

rep_months = pd.concat(runs, ignore_index=True)[
    ["strategy", "universe", "lam", "date", "month", "gross", "fee", "net", "turnover", "held", "effective_n"]]
acc = pd.concat([c for c in checks if c["table"].iloc[0].startswith("Backtest")])
print(f"{rep_months.groupby(['strategy', 'universe', 'lam']).ngroups} runs; "
      f"accounting checks: {len(acc)} run, {(acc['status'] == 'FAIL').sum()} failed")
rm_columns = variables.check_table(rep_months, "Replication Months")
checks.append(rm_columns.assign(table="Replication Months"))
print("Column checks:", rm_columns["status"].value_counts().to_dict())
if (acc["status"] == "FAIL").any() or (rm_columns["status"] == "FAIL").any():
    raise RuntimeError("A backtest check failed.")

## 7. Results next to Table 3

The first table covers the first 70 test months, the window the paper's turnover figures point to (Replication Spec). The full 81 months are saved alongside.

In [ ]:
def summarize(rows):
    return pd.DataFrame([backtest.metrics(g).rename((s, u, l))
                         for (s, u, l), g in rows.groupby(["strategy", "universe", "lam"])])

full = summarize(rep_months)
first70 = summarize(rep_months[rep_months["month"] <= config.PAPER_LAST_MONTH])
metric_names = ["ann_return", "ann_vol", "sharpe", "max_drawdown", "turnover"]
paper = pd.concat([backtest.paper_rows("PTO").assign(strategy="Plain PTO"),
                   backtest.paper_rows("MVO").assign(strategy="Classic Markowitz")]
                  ).set_index(["strategy", "universe", "lam"])

def side_by_side(table):
    out = table[metric_names + ["effective_n"]].copy()
    out.index.names = ["strategy", "universe", "lam"]
    return out.join(paper.add_prefix("paper_"))

comparison70 = side_by_side(first70)
cols = ["turnover", "paper_turnover", "ann_vol", "paper_ann_vol", "sharpe", "paper_sharpe",
        "max_drawdown", "paper_max_drawdown", "effective_n"]
print("First 70 months")
print(comparison70[cols].round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, u in zip(axes, universes):
    for name, color in (("Plain PTO", "tab:red"), ("Classic Markowitz", "tab:blue")):
        s = comparison70.loc[(name, u)]
        ax.plot(s.index, s["turnover"], "o-", color=color, label=f"{name}, this replication")
        ax.plot(s.index, s["paper_turnover"], "s--", color=color, alpha=0.6, label=f"{name}, paper")
    ax.set_xscale("log")
    ax.set_xlabel("risk aversion λ")
    ax.set_title(f"Monthly turnover, {u}")
axes[0].set_ylabel("monthly turnover")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. Pass criteria

The four criteria from the Replication Spec, on ETF_A over the first 70 months:

1. Plain PTO turnover is at least 0.6 at every λ, and changes by less than 0.1 between λ = 0.1 and λ = 50.
2. Plain PTO turnover is at least three times classic Markowitz turnover at every λ from 10 up.
3. Classic Markowitz volatility falls at each step of the λ grid, and its turnover is below 0.3 at every λ.
4. Classic Markowitz volatility at λ = 10, 20 and 50 is within 30% of the paper's value.

Criteria 1 and 2 are H1. Criteria 3 and 4 confirm the optimizer and covariance are on the paper's scale. A failed criterion is reported as it stands; settings are not changed to make it pass.

In [ ]:
pto = comparison70.loc[("Plain PTO", "ETF_A")].sort_index()
mvo = comparison70.loc[("Classic Markowitz", "ETF_A")].sort_index()
hi = [l for l in config.LAMBDA_GRID if l >= 10]
ratio = (pto.loc[hi, "turnover"] / mvo.loc[hi, "turnover"])
vol_gap = (mvo.loc[hi, "ann_vol"] / mvo.loc[hi, "paper_ann_vol"] - 1).abs()
change = abs(pto.loc[50.0, "turnover"] - pto.loc[0.1, "turnover"])
criteria = pd.DataFrame([
    {"check": "1. PTO trades heavily at every λ",
     "status": "PASS" if (pto["turnover"] >= 0.6).all() and change < 0.1 else "FAIL",
     "detail": f"turnover {pto['turnover'].min():.3f} to {pto['turnover'].max():.3f}; "
               f"change from λ = 0.1 to 50: {change:.3f}"},
    {"check": "2. PTO trades 3x Markowitz at λ ≥ 10",
     "status": "PASS" if (ratio >= 3).all() else "FAIL",
     "detail": "ratios " + ", ".join(f"{r:.1f}" for r in ratio)},
    {"check": "3. Markowitz volatility falls, turnover < 0.3",
     "status": "PASS" if (mvo["ann_vol"].diff().dropna() < 0).all() and (mvo["turnover"] < 0.3).all() else "FAIL",
     "detail": "volatility " + ", ".join(f"{v:.3f}" for v in mvo["ann_vol"])
               + f"; turnover up to {mvo['turnover'].max():.3f}"},
    {"check": "4. Markowitz volatility within 30% of paper",
     "status": "PASS" if (vol_gap <= 0.3).all() else "FAIL",
     "detail": "gaps " + ", ".join(f"{g:.0%}" for g in vol_gap) + " at λ = 10, 20, 50"},
])
print(criteria.to_string(index=False))
checks.append(criteria.assign(table="Pass criteria"))

## 9. What the PTO portfolio holds

The weights of plain PTO and classic Markowitz on ETF_A at λ = 20, month by month. A portfolio that jumps from one block of colour to another is switching most of its money between ETFs. The table gives the share of months in which one ETF holds more than 90% of the portfolio.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
colors = plt.cm.tab10(np.arange(len(config.ETF_A)))
for ax, name in zip(axes, ("Plain PTO", "Classic Markowitz")):
    w = weights_by_run[(name, "ETF_A", 20.0)].pivot(index="date", columns="ticker", values="target")[config.ETF_A]
    ax.stackplot(w.index, w.T.to_numpy(), labels=config.ETF_A, colors=colors, step="post")
    ax.set_title(f"{name}, ETF_A, λ = 20")
    ax.set_ylim(0, 1)
axes[0].set_ylabel("target weight")
axes[1].legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
plt.tight_layout()
plt.show()

single = []
for (name, u, lam), w in weights_by_run.items():
    top = w.groupby("date")["target"].max()
    top = top[top.index <= pd.Timestamp(config.PAPER_LAST_MONTH) - pd.offsets.MonthEnd(1)]
    single.append({"strategy": name, "universe": u, "lam": lam, "one ETF above 90%": (top > 0.9).mean()})
print(pd.DataFrame(single).pivot_table(index=["strategy", "universe"], columns="lam",
                                       values="one ETF above 90%").round(2).to_string())

## 10. Save

Saves the forecasts, the monthly results, the summary next to the paper and every check.

In [ ]:
fc.to_parquet(proc / config.FILES["forecasts"], index=False)
rep_months.to_parquet(proc / config.FILES["replication_months"], index=False)
results = pd.concat([side_by_side(full).assign(window="81 months"), comparison70.assign(window="70 months")])
results.reset_index().to_csv(DIRS["reports"] / config.FILES["replication_results"], index=False)
all_checks = pd.concat(checks, ignore_index=True)
all_checks.to_csv(DIRS["reports"] / config.FILES["replication_checks"], index=False)
print("Saved: PTO Forecasts, Replication Months, Replication Results, Replication Checks")

## 11. Summary

Key figures for the process log. After the run, save the notebook (File > Save a copy in GitHub) so the outputs are kept.

In [ ]:
passed = int((all_checks["status"] == "PASS").sum())
failed = int((all_checks["status"] == "FAIL").sum())
a = comparison70.xs("ETF_A", level="universe")
print(f"Forecasts:      {len(fc):,} ({len(calendar)} months, both universes)")
print(f"ETF_A, 70 months, plain PTO turnover:        {a.loc['Plain PTO', 'turnover'].min():.2f} to {a.loc['Plain PTO', 'turnover'].max():.2f} (paper 0.83 to 0.84)")
print(f"ETF_A, 70 months, classic Markowitz turnover: {a.loc['Classic Markowitz', 'turnover'].min():.2f} to {a.loc['Classic Markowitz', 'turnover'].max():.2f} (paper 0.05 to 0.25)")
print(f"Pass criteria:  {', '.join(criteria['status'])}")
print(f"Checks:         {passed} passed, {failed} failed")